## Nous allons faire cette analyse en répondant à quelques questions métiers dans le sens de l'étude de notre dataset afin de savoir
### Quel est le profil d'un patient ayant un rique élevé de faire un AVC ?
Pour cela, nous allons comparer la population de stroke = 0 et la population de stroke = 1

In [ ]:
# Bibliothèques
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

In [ ]:
# Importons notre dataset
df_clean = pd.read_csv("../data/healthcare_stroke_dataset_clean.csv")

display(df_clean.head())

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


#### Quel est le profil moyen des 2 groupes ?

In [ ]:
# Nous allons résumé les profils en 2 groupes avec des facteurs précis
profil_stroke = df_clean.groupby("stroke").agg({"age":"mean", 
                                "avg_glucose_level":"mean",
                                  "bmi": "mean",
                                  "hypertension": "mean",
                                  "heart_disease" : "mean"
                                  }).round(2)

profil_stroke

,age,avg_glucose_level,bmi,hypertension,heart_disease
stroke,,,,,
0,41.97,104.79,28.80,0.09,0.05
1,67.73,132.54,30.09,0.27,0.19


### Quels facteurs présentent les plus fortes différences ?
Nous verrons quelles variables changent le plus lorsqu'un patient à l'AVC

In [10]:
profil_stroke.loc[1] - profil_stroke.loc[0]

age                  25.76
avg_glucose_level    27.75
bmi                   1.29
hypertension          0.18
heart_disease         0.14
dtype: float64

Il y a trois facteurs qui ressortent considérablement:

- le facteur `âge` qui domine avec un écart de près de 26 ans.
- le facteur `glycémie`: les patients ayant subi un AVC présentent une glycémie moyenne beaucoup plus élevée. Avec une écart de près de 28 mg/dL.
- le facteur `hypertension` : on a une importance différente 27 % des patients victimes d'un AVC sont hypertendus contre seulement 9 % chez les autres. 

La maladie cardiaque vient juste derrière.

L'IMC semble avoir un effet plus limité.

### Quelle est la répartition de l'AVC selon les catégories ?
#### Gender


In [14]:
pd.crosstab(df_clean["gender"], df_clean["stroke"],normalize="index").round(3)*100

stroke,0,1
gender,,
Female,95.3,4.7
Male,94.9,5.1


La différence est très faible.

Nous dirons que le sexe ne semble pas être un facteur majeur dans ce jeu de données.

#### Mariage

In [16]:
pd.crosstab(df_clean["ever_married"], df_clean["stroke"],normalize="index").round(3)*100

stroke,0,1
ever_married,,
No,98.3,1.7
Yes,93.4,6.6


Nous dirions à première vue que les personnes mariées semblent présenter davantage d'AVC.

Cette observation nous amènera à la confusion car les personnes mariées sont en moyenne plus âgées.

Et nous savons déjà que : l'âge est LE principal facteur.

Donc ici on parle probablement d'un facteur de confusion.

#### Type d'emploi

In [24]:
pd.crosstab(df_clean["work_type"], df_clean["stroke"] , normalize="index").round(3)*100

stroke,0,1
work_type,,
Govt_job,95.0,5.0
Never_worked,100.0,0.0
Private,94.9,5.1
Self-employed,92.1,7.9
children,99.7,0.3


Nous dirions à première vue que les travailleurs indépendants semblent présenter davantage d'AVC.

N'oublions pas que les travailleurs indépendants sont souvent plus âgés.

Il est donc probable que l'âge explique une partie de cette différence.

Certainement un autre un facteur de confusion

#### Tabagisme

In [25]:
pd.crosstab(df_clean["smoking_status"], df_clean["stroke"] , normalize="index").round(3)*100

stroke,0,1
smoking_status,,
Unknown,97.0,3.0
formerly smoked,92.1,7.9
never smoked,95.2,4.8
smokes,94.7,5.3


Le taux d'AVC le plus élevé est observé chez les anciens fumeurs (7,9 %).

En revanche, n'oublions qui dit ancien fumeur dit plus âgé que les autres catégories.

Une analyse multivariée serait nécessaire pour isoler l'effet propre du tabagisme.

#### Résidence

In [18]:
pd.crosstab(df_clean["Residence_type"], df_clean["stroke"],normalize="index").round(3)*100

stroke,0,1
Residence_type,,
Rural,95.5,4.5
Urban,94.8,5.2


Nous avons une très faible différence entre les 2 groupes.

Le type de résidence ne semble pas fortement associé à la survenue d'un AVC dans ce jeu de données.

### Pour répondre à notre question du profil d'une personne susceptible de faire un AVC
#### Nous classerions nos facteurs de risques observés ainsi selon leur importance :
Si je devais faire un classement des facteurs observés :

- Âge
- Glycémie
- Hypertension
- Maladie cardiaque
- Tabagisme
- IMC
- Sexe
- Résidence

### Tableau de synthèse

In [26]:
df_clean.columns


Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

In [30]:
# Réalisons notre tableau de synthèse
table_synthese = pd.DataFrame({"Facteur" : ["Âge moyen", "Glycémie moyenne", "IMC moyen", "Hypertension", "Maladie Cardiaque" ],
  
  "Sans AVC" : [profil_stroke.loc[0, "age"], profil_stroke.loc[0, "avg_glucose_level"] , 
                profil_stroke.loc[0, "bmi"], f"{profil_stroke.loc[0, "hypertension"]*100:.0f} %", 
                f"{profil_stroke.loc[0, "heart_disease"]*100:.0f} %"],

  "Avec AVC" : [profil_stroke.loc[1, "age"], profil_stroke.loc[1, "avg_glucose_level"] , 
                profil_stroke.loc[1, "bmi"], f"{profil_stroke.loc[1, "hypertension"]*100:.0f} %", 
                f"{profil_stroke.loc[1, "heart_disease"]*100:.0f} %"],

  "Écart": [ round(profil_stroke.loc[1, "age"]           -    profil_stroke.loc[0, "age"],2),
        round(profil_stroke.loc[1, "avg_glucose_level"]  -    profil_stroke.loc[0, "avg_glucose_level"],2),
        round(profil_stroke.loc[1, "bmi"]                -    profil_stroke.loc[0, "bmi"],2),
        f" + {round((profil_stroke.loc[1,"hypertension"]   -    profil_stroke.loc[0,"hypertension"])*100,1)} pts",
        f" + {round((profil_stroke.loc[1,"heart_disease"]  -    profil_stroke.loc[0,"heart_disease"])*100,1)} pts"],

"Interprétation": [
        "Association très forte",
        "Association forte",
        "Association faible",
        "Association forte",
        "Association forte" ]
})

display(table_synthese)


,Facteur,Sans AVC,Avec AVC,Écart,Interprétation
0,Âge moyen,41.97,67.73,25.76,Association très forte
1,Glycémie moyenne,104.79,132.54,27.75,Association forte
2,IMC moyen,28.8,30.09,1.29,Association faible
3,Hypertension,9 %,27 %,+ 18.0 pts,Association forte
4,Maladie Cardiaque,5 %,19 %,+ 14.0 pts,Association forte
